# RecycleVision — fine-tuning on a free GPU

Free Colab gives you an NVIDIA T4, which trains this dataset in roughly an hour
where a laptop CPU would take most of a day.

**Before running anything: Runtime → Change runtime type → T4 GPU.**
Without that the whole notebook runs on CPU and there is no point.

Colab disconnects after a period of inactivity and caps sessions at a few
hours, so the last cell copies the trained weights to your Google Drive. If
you skip it and the session dies, the training is gone.

## 1. Check you actually got a GPU

If this errors or prints nothing, go back and change the runtime type.

In [ ]:
!nvidia-smi

## 2. Get the code

In [ ]:
!git clone https://github.com/encore488/recycle_vision.git
%cd recycle_vision
!pip install -q ultralytics pyyaml pillow

## 3. Get WaRP

Two routes. The Kaggle API is faster and avoids a multi-gigabyte upload from
your laptop.

**Your Kaggle token:** kaggle.com → your avatar → Settings → API → *Create New
Token*. That downloads `kaggle.json`. Run the cell below and upload it when
prompted.

In [ ]:
from google.colab import files
print("Upload your kaggle.json")
files.upload()

!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle datasets download -d parohod/warp-waste-recycling-plant-dataset -p /content/warp --unzip
!ls /content/warp

**Alternative, if the Kaggle route gives you trouble:** put your already-imported
`datasets/warp` folder in Google Drive and mount it instead. It is a large
upload, but it skips the API entirely.

```python
from google.colab import drive
drive.mount('/content/drive')
!cp -r "/content/drive/MyDrive/warp" /content/warp
```

## 4. Translate WaRP's classes into ours

WaRP labels 28 item classes; this project routes bins. `mappings/warp.yaml`
translates one to the other and records what the translation cannot express.

Find the real path to WaRP's `data.yaml` first — the archive layout varies, and
guessing it is the most likely thing to go wrong in this notebook.

In [ ]:
!find /content/warp -name "data.yaml" -maxdepth 4

In [ ]:
# Set this to whatever the cell above printed.
WARP_YAML = "/content/warp/Warp-D-yolo/data.yaml"

!python scripts/import_dataset.py {WARP_YAML} \
    --mapping mappings/warp.yaml --out datasets/warp --link

## 5. Measure before training

Worth the two minutes: without a before number, the after number means nothing.
Expect roughly 56% fair precision at 52.6% recall, at conf 0.01.

In [ ]:
!python scripts/zeroshot_eval.py --data datasets/warp/data.yaml --sweep

## 6. Train

`--device auto` finds the T4. About an hour for 100 epochs on ~3k images at
960px; drop `--epochs` if you want a first result sooner.

If you hit a CUDA out-of-memory error, lower `--batch` to 4, then 2.

In [ ]:
!python train.py --data datasets/warp/data.yaml --epochs 100 --batch 8 --device auto

## 7. Evaluate

This reports mAP *and* routing accuracy — whether predictions reach the right
bin, which is the number this project actually cares about. Confusing two
classes that share a bin costs nothing.

In [ ]:
import glob
import os

# WaRP is labelled with boxes, so train.py trains a detection model and
# ultralytics writes to runs/detect/. A polygon dataset would land in
# runs/segment/. Glob both rather than assuming which run just happened.
runs = sorted(glob.glob("runs/*/*/weights/best.pt"), key=os.path.getmtime)
assert runs, "no weights found — did the training cell finish?"
WEIGHTS = runs[-1]
print("evaluating", WEIGHTS)

!python scripts/evaluate.py --weights {WEIGHTS} --data datasets/warp/data.yaml

## 8. Save the weights before the session dies

**Do not skip this.** Colab reclaims the machine and takes everything with it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p "/content/drive/MyDrive/recyclevision"
!cp {WEIGHTS} "/content/drive/MyDrive/recyclevision/best_model.pt"
!ls -la "/content/drive/MyDrive/recyclevision/"

## 9. Back on your laptop

```bash
mkdir -p models
cp ~/Downloads/best_model.pt models/best_model.pt
```

The app prefers `models/best_model.pt` over stock weights automatically — no
code change, no config. Then compare like for like:

```bash
python scripts/evaluate.py --weights models/best_model.pt \
    --data datasets/warp/data.yaml
```

**What would count as success:** recall holding above 50% at a sane threshold
like 0.25 rather than 0.01. The zero-shot model already finds these objects; it
scores them terribly. Training is meant to fix the ranking, not the eyes.

**What this will not fix:** WaRP has no tableware, no organics, no film. The
three cases the README opens with still need your own footage.